# Adversarial Lab — Google Colab (CPU gratis)

Ejecuta el ciclo sintético del laboratorio: API con IDOR, repo con canario, agente con inyección y fallo/restore.

**Solo datos sintéticos.** No hay clientes, no hay claves de proveedor y no se toca ningún sistema externo.
Colab no es un sandbox de seguridad: no ejecutes aquí código hostil.

In [ ]:
import os, pathlib
if not pathlib.Path('/content/repo').exists():
    !git clone --depth 1 https://github.com/uniscrolliabrain-cloud/agntia-ciberseguridad.git /content/repo
%cd /content/repo
!python -m pip install -q -e adversarial_lab
print('instalado')

In [ ]:
!PYTHONPATH=adversarial_lab/src python -m agntia_adversarial.cli doctor

In [ ]:
import json, os, pathlib, subprocess, sys

env = dict(os.environ, PYTHONPATH='adversarial_lab/src', ADVLAB_DATA_DIR='/content/advlab-var')
proc = subprocess.run(
    [sys.executable, '-m', 'agntia_adversarial.cli', 'run', '--tenant', 'tenant-colab'],
    cwd='/content/repo', env=env, capture_output=True, text=True, timeout=300,
)
print(proc.stdout or proc.stderr)
payload = json.loads(proc.stdout) if proc.returncode == 0 else {}
report = pathlib.Path(payload.get('report', ''))
print(report.read_text() if report.exists() else 'sin informe')

In [ ]:
!cd /content && zip -qr advlab-evidence.zip advlab-var/reports advlab-var/evidence
from google.colab import files
files.download('/content/advlab-evidence.zip')

## Límites

- Resultado esperado: **PARTIAL**, con las siete herramientas externas en `BLOCKED`.
- Sin proveedor real: el ciclo usa `mock`.
- La evidencia queda cifrada en `/content/advlab-var`; al cerrar la sesión de Colab desaparece.
- Colab es un entorno de desarrollo compartido, no una frontera de aislamiento.